# 02 - Spatial Baseline + Hybrid (CV-Injected)


In [3]:
import sys
from pathlib import Path

SRC_DIR = (Path.cwd().resolve() / '..' / 'src').resolve()
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

# Hard-reload local package modules so notebook always sees latest edits.
for name in list(sys.modules.keys()):
    if name == 'course_project' or name.startswith('course_project.'):
        del sys.modules[name]

from course_project.config import ExperimentConfig
from course_project.runner import run_experiment



In [4]:
# Train spatial baseline + hybrid with 3 repeats each, compare mean best rollout R2
import numpy as np
import torch
import pandas as pd
import json

common_cfg = dict(
    train_dataset='../data/2340_dePablo_networks_OOL_undirected_train1.pt',
    val_dataset='../data/2340_dePablo_networks_OOL_undirected_val100.pt',
    output_root='../results',
    device='cuda',
    pos_dim=2,
    history=1,
    limit=40,
    hidden_size=64,
    n_layers=2,
    learning_rate=1e-4,
    learning_rate_decay=0.997,
    epochs=200,
    val_every=10,
    rollout_steps=100,
    rollout_every=10,
    cv_eval_every=10,
    cv_pratio_target='box',
    train_rollout_steps=2,
    train_rollout_loss_decay=1,
)

best_cv_info = json.loads(Path('../results/cv_transformer_best_cv_selection.json').read_text())
best_cv_ckpt = Path(best_cv_info['saved_checkpoint_path'])
print('Using CV checkpoint:', best_cv_ckpt)
print('Best CV fit R2 across runs:', float(best_cv_info['best_cv_fit_r2']), 'run:', best_cv_info['run_name'], 'epoch:', int(best_cv_info['best_cv_epoch']))

repeats = 3
seed_base = 123
rows = []

for i in range(repeats):
    seed = seed_base + i

    cfg_hybrid = ExperimentConfig(
        run_name=f'hybrid_r{i+1}',
        model_type='hybrid',
        hybrid_global_only_epochs=0,
        seed=seed,
        model_extras={
            'num_mlp': 3,
            'cv_checkpoint_path': str(best_cv_ckpt),
            'cv_inject_scale_init': 1,
        },
        **common_cfg,
    )

    cfg_spatial = ExperimentConfig(
        run_name=f'spatial_baseline_r{i+1}',
        model_type='spatial',
        seed=seed,
        model_extras={
            'num_mlp': 3,
        },
        **common_cfg,
    )

    m_hybrid = run_experiment(cfg_hybrid)
    m_spatial = run_experiment(cfg_spatial)

    hybrid_ckpt = torch.load(Path(common_cfg['output_root']) / cfg_hybrid.run_name / 'final_checkpoint.pt', map_location='cpu', weights_only=False)
    hybrid_stats = torch.load(Path(common_cfg['output_root']) / cfg_hybrid.run_name / 'train_stats.pt', map_location='cpu', weights_only=False)

    m_hybrid['repeat'] = i + 1
    m_hybrid['cv_inject_scale'] = float(hybrid_ckpt['model_state_dict']['cv_inject_scale'])
    m_hybrid['cv_node_gate_mean'] = float(np.asarray(hybrid_stats['cv_node_gate_mean'], dtype=float)[-1])

    m_spatial['repeat'] = i + 1
    m_spatial['cv_inject_scale'] = np.nan
    m_spatial['cv_node_gate_mean'] = np.nan

    rows.append(m_spatial)
    rows.append(m_hybrid)

runs = pd.DataFrame(rows)

runs_summary = runs[[
    'repeat',
    'run_name',
    'model_type',
    'best_rollout_epoch',
    'best_rollout_r2',
    'rollout_r2',
    'rollout_pearson_r',
    'rollout_pos_mse',
    'cv_fit_r2',
    'cv_inject_scale',
    'cv_node_gate_mean',
]].sort_values(['model_type', 'repeat'])

compare = runs.groupby('model_type', as_index=False).agg(
    mean_best_rollout_r2=('best_rollout_r2', 'mean'),
    std_best_rollout_r2=('best_rollout_r2', 'std'),
    mean_rollout_r2=('rollout_r2', 'mean'),
)

print('Per-run results:')
runs_summary



Using CV checkpoint: ../results/cv_transformer_best_cv_checkpoint.pt
Best CV fit R2 across runs: 0.9290038899079617 run: cv_transformer_fixed_r04 epoch: 10
[run] hybrid_r1 model=hybrid device=cuda output=../results/hybrid_r1
[run] training...
[train] autoregressive loss steps=2 decay=1
[ep  10/200] tr=2.57 va=1.72 lr=9.7e-05 roll=r2=0.106 p=0.866 mse=6.44e-05 (100/100) cv_gate=0.999 gmean=0.694 cv=|p|=0.964 r2=0.929 (n=100) t=53.6s
[ep  20/200] tr=2.53 va=1.73 lr=9.42e-05 roll=r2=0.339 p=0.877 mse=6.56e-05 (100/100) cv_gate=0.999 gmean=0.736 cv=|p|=0.964 r2=0.929 (n=100) t=53.1s
[ep  30/200] tr=2.49 va=1.76 lr=9.14e-05 roll=r2=0.408 p=0.854 mse=7.12e-05 (100/100) cv_gate=0.998 gmean=0.672 cv=|p|=0.964 r2=0.929 (n=100) t=53s
[ep  40/200] tr=2.47 va=1.76 lr=8.87e-05 roll=r2=0.348 p=0.768 mse=7.05e-05 (100/100) cv_gate=0.997 gmean=0.617 cv=|p|=0.964 r2=0.929 (n=100) t=53.2s
[ep  50/200] tr=2.47 va=1.76 lr=8.61e-05 roll=r2=0.381 p=0.818 mse=6.89e-05 (100/100) cv_gate=0.997 gmean=0.668 cv=|

KeyboardInterrupt: 

In [ ]:
print('Mean comparison (use this to compare models):')
compare



Mean comparison (use this to compare models):


,model_type,mean_best_rollout_r2,std_best_rollout_r2,mean_rollout_r2
0,hybrid,0.859994,0.056842,0.859993
1,spatial,0.883423,0.038738,0.883423
